In [12]:
import wrds
db = wrds.Connection(wrds_username='liaojy')

Loading library list...
Done


In [3]:
db.describe_table(library='crsp', table='msf')

Approximately 5153763 rows in crsp.msf.


,name,nullable,type,comment
0,cusip,True,VARCHAR(8),CUSIP Header
1,permno,True,INTEGER,PERMNO
2,permco,True,INTEGER,PERMCO
3,issuno,True,INTEGER,Nasdaq Issue Number
4,hexcd,True,SMALLINT,Exchange Code Header
5,hsiccd,True,INTEGER,Standard Industrial Classification Code Header
6,date,True,DATE,Date of Observation
7,bidlo,True,"NUMERIC(11, 5)",Bid or Low Price
8,askhi,True,"NUMERIC(11, 5)",Ask or High Price
9,prc,True,"NUMERIC(11, 5)",Price or Bid/Ask Average


In [6]:
query = """SELECT permno, date, ret, prc, shrout
    FROM crsp.msf
    WHERE date >= '2024-01-01' AND date <= '2024-12-31'
    """
sample = db.raw_sql(query, date_cols=['date'])
print(sample.head(15))
print("shape:", sample.shape)

    permno       date       ret        prc     shrout
0    10026 2024-01-31 -0.047326     159.23    19367.0
1    10028 2024-01-31 -0.104938       4.35    26700.0
2    10032 2024-01-31 -0.124017      94.72    27612.0
3    10044 2024-01-31 -0.130435        4.0     6315.0
4    10065 2024-01-31  0.025409      18.16   120810.0
5    10066 2024-01-31 -0.159292       2.85    11784.0
6    10104 2024-01-31  0.063265      111.7  2748922.0
7    10107 2024-01-31  0.057281  397.57999  7430436.0
8    10113 2024-01-31 -0.004447      55.97      450.0
9    10138 2024-01-31  0.007057     108.45   223938.0
10   10145 2024-01-31 -0.035525  202.25999   652182.0
11   10158 2024-01-31  -0.35491      20.43    34235.0
12   10200 2024-01-31  0.053393  189.39999    55766.0
13   10207 2024-01-31    -0.035       7.72    29847.0
14   10220 2024-01-31  0.061905      81.48    91513.0
shape: (116119, 5)


In [8]:
import numpy as np
sample['mktcap'] = sample['prc'].abs() * sample['shrout']
print(sample[['permno','date','prc','shrout','mktcap']].head())
print("\n2024-01 5 largest market value:")
jan = sample[sample['date']=='2024-01-31']
print(jan.nlargest(5,'mktcap')[['permno','prc','mktcap']])

   permno       date     prc    shrout      mktcap
0   10026 2024-01-31  159.23   19367.0  3083807.41
1   10028 2024-01-31    4.35   26700.0    116145.0
2   10032 2024-01-31   94.72   27612.0  2615408.64
3   10044 2024-01-31     4.0    6315.0     25260.0
4   10065 2024-01-31   18.16  120810.0   2193909.6

2024-01 5 largest market value:
      permno        prc            mktcap
7      10107  397.57999  2954192670.57564
929    14593  184.39999  2847482701.98119
7747   84788      155.2      1612121531.2
7930   86580  615.27002     1516025329.28
486    13407  390.14001   858327138.86049


In [4]:
query = """
SELECT permno, date, ret, prc, shrout
FROM crsp.msf
WHERE date >= '2020-01-01' AND date <= '2024-12-31'
"""
df = db.raw_sql(query, date_cols=['date'])
df = df.sort_values(['permno','date']).reset_index(drop=True)

print(df.shape)
print(df.head())

(546882, 5)
   permno       date       ret        prc   shrout
0   10026 2020-01-31 -0.100016     165.84  18919.0
1   10026 2020-02-28  -0.03027  160.82001  18919.0
2   10026 2020-03-31 -0.244031      121.0  18888.0
3   10026 2020-04-30  0.049835     127.03  18888.0
4   10026 2020-05-29  0.012596     128.63  18888.0


In [8]:
import numpy as np
df['logret']=np.log(1+df['ret'])
df['mom_12_2']=df.groupby('permno')['logret'].transform(lambda x: x.shift(1).rolling(window=11).sum())
print(df[['permno','date','ret','mom_12_2']].head(20))


    permno       date       ret  mom_12_2
0    10026 2020-01-31 -0.100016       NaN
1    10026 2020-02-28  -0.03027       NaN
2    10026 2020-03-31 -0.244031       NaN
3    10026 2020-04-30  0.049835       NaN
4    10026 2020-05-29  0.012596       NaN
5    10026 2020-06-30 -0.007191       NaN
6    10026 2020-07-31 -0.031464       NaN
7    10026 2020-08-31  0.104118       NaN
8    10026 2020-09-30 -0.036668       NaN
9    10026 2020-10-30  0.039727       NaN
10   10026 2020-11-30  0.072435       NaN
11   10026 2020-12-31  0.072598 -0.223327
12   10026 2021-01-29 -0.017442 -0.047865
13   10026 2021-02-26  0.039958 -0.034724
14   10026 2021-03-31 -0.007275  0.284211
15   10026 2021-04-30  0.048271  0.228277
16   10026 2021-05-28  0.066642  0.262902
17   10026 2021-06-30 -0.003058  0.334634
18   10026 2021-07-30 -0.057508  0.363541
19   10026 2021-08-31 -0.003772  0.205266


In [13]:
db.describe_table(library='comp',table='funda')

Approximately 941420 rows in comp.funda.


,name,nullable,type,comment
0,gvkey,True,VARCHAR(7),Global Company Key
1,datadate,True,DATE,Data Date
2,fyear,True,INTEGER,Data Year - Fiscal
3,indfmt,True,VARCHAR(13),Industry Format
4,consol,True,VARCHAR(3),Level of Consolidation - Company Annual Descri...
...,...,...,...,...
944,au,True,VARCHAR(9),Auditor
945,auop,True,VARCHAR(9),Auditor Opinion
946,auopic,True,VARCHAR(2),Auditor Opinion - Internal Control
947,ceoso,True,VARCHAR(2),Chief Executive Officer SOX Certification


In [15]:
query = """
    SELECT gvkey, datadate, fyear, ceq, pstk, txditc, seq, at, lt
    FROM comp.funda
    WHERE datadate >= '2020-01-01' AND datadate <= '2024-12-31'
    AND indfmt = 'INDL'
    AND datafmt = 'STD'
    AND popsrc = 'D'
    AND consol = 'C'
"""

comp = db.raw_sql(query, date_cols=['datadate'])

print(comp.head(15))
print("shape:", comp.shape)

     gvkey   datadate  fyear      ceq  pstk  txditc      seq       at       lt
0   001004 2020-05-31   2019    902.6   0.0     0.0    902.6   2079.0   1176.4
1   001004 2021-05-31   2020    974.4   0.0     9.5    974.4   1539.7    565.3
2   001004 2022-05-31   2021   1034.5   0.0    20.0   1034.5   1573.9    539.4
3   001004 2023-05-31   2022   1099.1   0.0    33.6   1099.1   1833.1    734.0
4   001004 2024-05-31   2023   1189.8   0.0    23.9   1189.8   2770.0   1580.2
5   001019 2020-12-31   2020   13.479   0.0   0.361   13.479    40.57   27.091
6   001045 2020-12-31   2020  -6867.0   0.0     9.0  -6867.0  62008.0  68875.0
7   001045 2021-12-31   2021  -7340.0   0.0     9.0  -7340.0  66467.0  73807.0
8   001045 2022-12-31   2022  -5799.0   0.0    10.0  -5799.0  64716.0  70515.0
9   001045 2023-12-31   2023  -5202.0   0.0     9.0  -5202.0  63058.0  68260.0
10  001045 2024-12-31   2024  -3977.0   0.0     9.0  -3977.0  61783.0  65760.0
11  001050 2020-12-31   2020  202.658   0.0    6.97 